# Generated single-grain shape grid

Visualize the exact shape meshes produced by `single_grain_coercivity.py` for the four shape variants used in material/shape comparisons.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / 'utils').is_dir():
    SINGLE_GRAIN_DIR = Path('python/experiments/single_grain').resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from single_grain_coercivity import build_shape_mesh, resolve_shape_spec
from utils.geometry import PrismMesh
from utils.geometry_plotting import plot_prism_mesh

SHAPES = ('cube', 'sphere', 'ellipsoid_x', 'ellipsoid_z')
RESOLUTIONS = (10, 12, 14)
OUTER_SIZE_M = 1.0
EXPORT = True
EXPORT_DIR = SINGLE_GRAIN_DIR / 'figures' / 'generated_shape_grid'

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.titlesize': 11,
    'figure.titlesize': 18,
})

In [ ]:
def cube_mesh(outer_size_m, n):
    side = float(outer_size_m)
    cell = side / int(n)
    axis = np.linspace(-side / 2.0 + cell / 2.0, side / 2.0 - cell / 2.0, int(n))
    centers = np.array(np.meshgrid(axis, axis, axis, indexing='ij')).reshape(3, -1).T
    dimensions = np.full_like(centers, cell, dtype=float)
    levels = np.zeros(len(centers), dtype=np.int64)
    root_bounds = np.array([[-side / 2.0, -side / 2.0, -side / 2.0], [side / 2.0, side / 2.0, side / 2.0]])
    return PrismMesh(
        centers=centers,
        dimensions=dimensions,
        levels=levels,
        shape='cube',
        shape_metadata={'side_length': side},
        root_bounds=root_bounds,
        target_tiles=int(n) ** 3,
    )


def generated_mesh(shape, n, outer_size_m=OUTER_SIZE_M):
    shape_spec = resolve_shape_spec(shape, None, outer_size_m)
    mesh = build_shape_mesh(shape_spec, int(n))
    if mesh is None:
        return shape_spec, cube_mesh(outer_size_m, int(n))
    return shape_spec, mesh


def display_shape_name(shape):
    return shape.replace('_', ' ').title()

## Shape meshes by spatial resolution

In [ ]:
fig, axes = plt.subplots(
    len(RESOLUTIONS), len(SHAPES),
    figsize=(16, 9),
    subplot_kw={'projection': '3d'},
    squeeze=False,
)

for row, n in enumerate(RESOLUTIONS):
    for col, shape in enumerate(SHAPES):
        shape_spec, mesh = generated_mesh(shape, n)
        ax = axes[row, col]
        plot_prism_mesh(
            mesh,
            fig=fig,
            ax=ax,
            color_by_level=True,
            cmap='viridis',
            alpha=0.52,
            wireframe=True,
            hide_axes=True,
            max_plot_tiles=None,
            elevation=22.0,
            azimuth=40.0,
        )
        bounds_span = mesh.root_bounds[1] - mesh.root_bounds[0]
        zoom = 1.0 if shape_spec.variant == 'cube' else 1.35
        try:
            ax.set_box_aspect(bounds_span, zoom=zoom)
        except TypeError:
            ax.set_box_aspect(bounds_span)
        ax.set_title(display_shape_name(shape_spec.variant) if row == 0 else '', pad=-2, fontsize=13, fontweight='bold')
        ax.text2D(
            0.04, 0.88, f'{mesh.achieved_tiles} squares',
            transform=ax.transAxes,
            va='top',
            ha='left',
            fontsize=9.5,
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.72, 'pad': 1.5},
        )
        if col == 0:
            ax.text2D(
                -0.06, 0.5, f'n={n}',
                transform=ax.transAxes,
                rotation=90,
                va='center',
                ha='center',
                fontsize=13,
                fontweight='bold',
            )

fig.subplots_adjust(left=0.035, right=0.995, top=0.965, bottom=0.015, wspace=-0.08, hspace=-0.10)

if EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(EXPORT_DIR / 'generated_shape_mesh_grid.png', dpi=200, facecolor='white')

plt.show()